# Heart Disease Prediction

An educational machine-learning project that compares Logistic Regression and Random Forest models for binary heart-disease classification using synthetic clinical-style data.

> **Data and medical disclaimer:** The included data is synthetic/fictional and does not represent real patients. This project is for educational purposes only and must not be used for medical diagnosis or clinical decision-making.


## Project workflow

1. Load the included `heart.csv` dataset.
2. Explore the data, class balance, correlations, and missing values.
3. Split the data into stratified training and test sets.
4. Standardize the input features.
5. Train Logistic Regression and Random Forest classifiers.
6. Compare accuracy and ROC AUC.
7. Review the confusion matrix, classification report, ROC curve, and Random Forest feature importance.

Run the notebook cells from top to bottom.

In [ ]:

# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')


## 2) Loading the data
Make sure `heart.csv` is in the same directory as this notebook. rename it If needed.

In [ ]:
from pathlib import Path
import os

#Using path
file_path = Path('heart.csv')

# Check if file exists
if not file_path.exists():
    raise FileNotFoundError(f"Please place 'heart.csv' file at: {file_path}")

# Load dataset from local file
data = pd.read_csv(file_path)
print("Loaded 'heart.csv' successfully. Shape:", data.shape)
data.head()

## 3) Basic exploration
Check data types, summary statistics, and missing values.

In [ ]:

# Basic info
data.info()

# Summary statistics
display(data.describe().T)

# Missing values per column
display(data.isnull().sum())


## 4) Visualizations
Class balance and feature correlations.

In [ ]:

# Target distribution (assumes 'target' column exists where 1 = disease, 0 = no disease)
if 'target' in data.columns:
    plt.figure(figsize=(6,4))
    sns.countplot(x='target', data=data)
    plt.title('Heart Disease Distribution (1 = Disease, 0 = No Disease)')
    plt.show()
else:
    print("No 'target' column found. Please ensure the dataset has a 'target' column.")

# Correlation heatmap (numeric columns)
plt.figure(figsize=(12,10))
sns.heatmap(data.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap')
plt.show()


## 5) Preprocessing
Separate features and target, train/test split, and feature scaling.

In [ ]:
# Verify target exists
if 'target' not in data.columns:
    print("Warning: Adding 'target' column from original dataset...")
    data_path = Path(r'heart.csv')
    data = pd.read_csv(data_path)
assert 'target' in data.columns, "Dataset must contain a 'target' column with 0/1 values."

# Separate features and target
X = data.drop('target', axis=1)
y = data['target']

# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

# Standardize numeric features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 6) Train models
Train Logistic Regression and Random Forest and compare afterwards

In [ ]:

# Logistic Regression
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train)
log_pred = log_model.predict(X_test_scaled)

# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)

# Print basic accuracy
print("Logistic Regression Accuracy:", accuracy_score(y_test, log_pred))
print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))


## 7) Evaluation
Detailed metrics and confusion matrix for the stronger model.

In [ ]:

# Compare and choose the better model based on accuracy and ROC AUC
models = {
    'Logistic Regression': (log_model, log_pred),
    'Random Forest': (rf_model, rf_pred)
}

for name, (model, pred) in models.items():
    try:
        roc = roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:,1])
    except Exception:
        roc = None
    acc = accuracy_score(y_test, pred)
    print(f"{name}: Accuracy = {acc:.4f}, ROC AUC = {roc if roc is not None else 'N/A'}")

# Use Random Forest's predictions for confusion matrix and classification report (as example)
best_model = rf_model
best_pred = rf_pred

print('\nClassification Report (Random Forest):')
print(classification_report(y_test, best_pred))

# Confusion matrix
cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Random Forest')
plt.show()

# ROC Curve
probs = best_model.predict_proba(X_test_scaled)[:,1]
fpr, tpr, thresholds = roc_curve(y_test, probs)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc_score(y_test, probs):.3f})')
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Random Forest')
plt.legend()
plt.show()


## 8) Feature importance on Random Forest
Which features influenced the Random Forest predictions most?

In [ ]:

# Feature importance
importances = rf_model.feature_importances_
feat_imp = pd.DataFrame({'feature': X.columns, 'importance': importances}).sort_values(by='importance', ascending=False)
display(feat_imp.head(15))

# Plot top 10 features
plt.figure(figsize=(8,5))
sns.barplot(x='importance', y='feature', data=feat_imp.head(10))
plt.title('Top 10 Feature Importances (Random Forest)')
plt.show()


## Notes on the results

On the included 50-row educational sample and the current 20% stratified holdout, the Random Forest model achieved 0.80 accuracy and 0.88 ROC AUC, compared with 0.60 accuracy and 0.68 ROC AUC for Logistic Regression. Because the test set contains only 10 rows, these metrics are illustrative rather than a reliable estimate of real-world clinical performance.

The model is not a medical device and should not be used to diagnose or make healthcare decisions.